# Use Case Description
Automating the prioritization of critical alerts from a large volume of security alerts is essential because it enables security teams to focus on the most urgent and impactful threats quickly. Without automation, manually sorting through numerous alerts can lead to delays, missed critical incidents, and inefficient use of resources. Automation using LLMs improves response times, reduces alert fatigue, and ensures that critical vulnerabilities are addressed promptly, enhancing overall security posture and operational efficiency.


## Model used for this use case
Both Instruct Model and Reasoning Model would be suitable for this task. In this example, we used Reasoning Model. <br>

## SetUp

We provide four different setup methods in this example. 

1. [Quickstart with Transformers](https://github.com/cisco-foundation-ai/cookbook/blob/main/1_quickstarts/Preview_Quickstart_reasoning_model.ipynb)
2. [Deploy on Sagemaker](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/sagemaker/foundation_sec_8b_reasoning/deploy.ipynb)
3. [Deploy on Baseten](https://github.com/cisco-foundation-ai/cookbook/tree/main/3_adoptions/deployment/baseten)
4. [Deploy on Bedrock](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/bedrock/foundation_sec_8b_reasoning/deploy.ipynb)

Make sure the helper file [inference_clients.py](https://github.com/cisco-foundation-ai/cookbook/blob/main/2_examples/inference_clients.py) is located in the same directory of this notebook and then: <br>
1. **uncomment** the preferred setup method in the cell below  <br> 
2. **Fill in** the necessary arguments based on your deployment type and run the following cell to initiate the helper.

In [1]:
from inference_clients import ReasoningModelClient
from IPython.display import display, Markdown

## Uncomment for Transformers Deployment
# client = ReasoningModelClient.from_transformers(
#     model_id="fdtn-ai/Foundation-Sec-8B-Reasoning"  # Hugging Face model name. For reasoning model: "fdtn-ai/Foundation-Sec-8B-Reasoning",
# )

## Uncomment for Sagemaker Deployment
# client = ReasoningModelClient.from_sagemaker(
#     endpoint_name=''  # Fill in your sagemaker deployment endpoint if applicable
# )

## Uncomment for Baseten Deployment
# client = ReasoningModelClient.from_baseten(
#     endpoint_url='',  # Fill in your baseten deployment endpoint if applicable. Example: https://XXXXX.api.baseten.co/environments/production/sync/v1/chat/completions,
#     api_key=''  # Fill in your baseten API key if applicable
# )

## Uncomment for Bedrock Deployment
# client = ReasoningModelClient.from_bedrock(
#     region='',
#     model_arn=''  # copy from deploy notebook or: aws bedrock list-imported-models
# )

## Alerts analysis

Here’s several mock security alerts to be analyzed

In [2]:
alerts = """
{
"alert_uuid": "3a4bd7e4-f8a2-4b7e-9d3b-7e2fa3f98b6c",
"timestamp": "2025-07-15T10:55:00Z",
"user": "security_admin",
"source_ip": "198.51.100.33",
"message": "The host at 198.51.100.33 has been permanently added to the ssh allowed list.",
"code": "SEC3007"
},
{
"alert_uuid": "a3f9d2b7-4c1e-4f8a-9d3b-7e2f1a6c8b9d",
"timestamp": "2025-07-15T09:45:12Z",
"user": "sysadmin",
"source_ip": "192.168.1.10",
"destination_ip": "10.0.0.5",
"message": "Log Error: Push error for subscription logs_sub_01: Failed to connect to 10.0.0.5: Connection timed out",
"code": "LOG1001"
},
{
"alert_uuid": "b9d3a4b7-e2f1-4c8b-9d3b-7e2fa3f98b6c",
"timestamp": "2025-07-15T10:20:45Z",
"user": "sysadmin",
"source_ip": "192.168.1.10",
"message": "Log Error: Subscription logs_sub_02: Log partition is full.",
"code": "LOG1002"
},
{
"alert_uuid": "d7e4f8a2-9b3c-4f1e-8a7d-2b6c9f1e3a4b",
"timestamp": "2025-07-15T10:05:33Z",
"user": "automation_bot",
"source_ip": "192.168.1.20",
"message": "Startup script backup_script.sh exited with error: Disk quota exceeded. System services failed to start, causing immediate operational outage.",
"code": "SYS2002"
},
{
"alert_uuid": "4c1e9d3b-7e2f-4f8a-9d3b-7e2fa3f98b6c",
"timestamp": "2025-07-15T10:45:55Z",
"user": "system",
"source_ip": "127.0.0.1",
"message": "Dependency cycle detected: processA -> processB -> processA.",
"code": "SYS2010"
},
{
"alert_uuid": "f1a6c8b9-d2b7-4c1e-9d3b-7e2fa3f98b6c",
"timestamp": "2025-07-15T10:15:00Z",
"user": "unknown",
"source_ip": "203.0.113.45",
"message": "The host at 203.0.113.45 has been added to the blocked list because of an SSH DOS attack.",
"code": "SEC3003"
},
{
"alert_uuid": "e2f1a6c8-b9d3-4c1e-9d3b-7e2fa3f98b6c",
"timestamp": "2025-07-15T10:35:00Z",
"user": "jdoe",
"source_ip": "198.51.100.22",
"message": "Account locked due to 5 failed login attempts. Last login attempt from 198.51.100.22.",
"code": "SEC3006"
},
{
"alert_uuid": "9d3b7e2f-a3f9-4c1e-8b6c-7e2fa3f98b6c",
"timestamp": "2025-07-15T10:40:20Z",
"user": "sysadmin",
"source_ip": "192.168.1.10",
"destination_ip": "10.0.0.6",
"message": "Subscription logs_sub_03: Timed out after 30 seconds sending data to syslog server syslog01 (10.0.0.6).",
"code": "LOG1003"
},
{
"alert_uuid": "7e2fa3f9-8b6c-4c1e-9d3b-7e2fa3f98b6c",
"timestamp": "2025-07-15T10:50:10Z",
"user": "reporting_service",
"source_ip": "192.168.1.40",
"message": "A failure occurred while building periodic report ‘Monthly_Security_Audit’.",
"code": "REP5001"
},
{
"alert_uuid": "c8b9d3a4-b7e2-4f1e-9d3b-7e2fa3f98b6c",
"timestamp": "2025-07-15T10:30:12Z",
"user": "app_service",
"source_ip": "192.168.1.30",
"message": "An application fault occurred: segmentation fault in module auth_handler.",
"code": "APP4005"
}
"""

In [3]:
def make_prompt(alerts):
   return f"""You are a security expert in charge of analyzing bunch of alerts and prioritizing critical ones.
    
    Go over the alerts given, and prioritize which ones are critical
    
    ## ALERTS
    {alerts}
    
    Respond with:
    - Which ones are critical?
    - Why are they crtical?
    - What should we do to address the alerts?

    """

In [4]:
reasoning, response = client.chat_generation(make_prompt(alerts))

print("=== Reasoning (chain-of-thought) ===")
display(Markdown(reasoning))
print()
display(Markdown("<mark>=== Answer ===</mark>"))
display(Markdown(response))

=== Reasoning (chain-of-thought) ===



My objective is to evaluate each alert provided and determine its criticality, explaining why and suggesting appropriate actions. I'll proceed by examining each alert individually, assessing its potential impact on system availability, integrity, or confidentiality.

Starting with **Alert 3a4bd7e4-f8a2-4b7e-9d3b-7e2fa3f98b6c**: This alert indicates that a host was permanently added to the SSH allowed list. While this could potentially allow unauthorized access if the addition was malicious, without further context confirming the legitimacy of the change, it represents a significant security risk. Unauthorized additions to trusted lists can lead to backdoor creation, so this warrants investigation.

Next, **Alert a3f9d2b7-4c1e-4f8a-9d3b-7e2f1a6c8b9d** reports a push error for subscription logs, specifically a connection timeout to 10.0.0.5. A failure in log transmission means critical audit trails might be lost, hindering incident response and compliance. This impacts forensic capabilities and regulatory adherence, making it a high-priority issue.

Moving to **Alert b9d3a4b7-e2f1-4c8b-9d3b-7e2fa3f98b6c**, where a log partition is full. When storage for logs runs out, new events cannot be recorded. This directly compromises the ability to monitor system activity, detect anomalies, and investigate incidents, thus severely impacting security monitoring and auditing functions.

Then there's **Alert d7e4f8a2-9b3c-4f1e-8a7d-2b6c9f1e3a4b**, indicating a startup script backup_script.sh exited with an error due to disk quota exceeded. This caused an immediate operational outage. An outage directly impacts business operations and service availability. The root cause being a disk quota issue suggests underlying resource management problems that need urgent attention to prevent recurrence.

**Alert 4c1e9d3b-7e2f-4f8a-9d3b-7e2fa3f98b6c** describes a dependency cycle detected between processes. Such cycles can lead to deadlocks, preventing applications from functioning correctly. This directly affects application stability and reliability, potentially causing intermittent failures or complete crashes, requiring immediate resolution.

Another critical alert is **Alert f1a6c8b9-d2b7-4c1e-9d3b-7e2fa3f98b6c**, where a host was added to the blocked list due to an SSH DoS attack. An SSH DoS attack aims to disrupt legitimate users' access, leading to denial-of-service conditions. This directly impacts system accessibility and availability, necessitating investigation into the attack vector and mitigation strategies.

**Alert e2f1a6c8-b9d3-4c1e-9d3b-7e2fa3f98b6c** shows an account lock due to five failed login attempts. Account lockouts are standard security measures against brute-force attacks. However, if this occurs frequently, it could indicate a targeted attack or a misconfiguration. It requires verification of the source IP and review of authentication policies.

**Alert 9d3b7e2f-a3f9-4c1e-8b6c-7e2fa3f98b6c** reports a timeout sending data to a syslog server. Similar to the previous log-related issues, this prevents crucial event logging. Without proper logging, security monitoring is severely impaired, making it a critical concern that needs prompt resolution.

Finally, **Alert 7e2fa3f9-8b6c-4c1e-9d3b-7e2fa3f98b6c** states a failure in building a periodic security report. Report generation failures mean that vital security information may not be produced, affecting compliance reporting and proactive security posture assessment. This impacts both internal visibility and external audits.

Based on this analysis, the most critical alerts are those that directly compromise system availability, integrity, or security monitoring capabilities. Alerts involving operational outages, denial-of-service conditions, severe application instability, and critical log failures fall into this category. The others, while important, represent more contained issues that require investigation rather than immediate emergency action.

Therefore, my priority list includes the disk quota outage, the SSH DoS attack, the dependency cycle, and the log transmission failures. Addressing these first is essential to restore normal operations and maintain security postures. The remaining alerts are important but can be addressed once the critical issues are resolved, focusing on understanding the root causes and implementing preventative measures.


<mark>=== Answer ===</mark>



### Critical Alerts Analysis

#### **Critical Alert 1: SEC3007**  
**UUID:** `3a4bd7e4-f8a2-4b7e-9d3b-7e2fa3f98b6c`  
**Why Critical:** Permanent addition to SSH allowed list bypasses security controls, enabling unauthorized access. This creates a backdoor for attackers, compromising confidentiality and integrity.  

**Action:**  
1. Immediately revoke the entry via SSH configuration management tools (e.g., `sudo nano /etc/ssh/sshd_config`).  
2. Audit all recent changes to SSH configurations using SIEM tools (e.g., Splunk, ELK Stack).  
3. Reset credentials for any accounts associated with the modified SSH config.  

---

#### **Critical Alert 2: SYS2002**  
**UUID:** `d7e4f8a2-9b3c-4f1e-8a7d-2b6c9f1e3a4b`  
**Why Critical:** Disk quota exceeded caused an *immediate operational outage*. This impacts business continuity and data availability.  

**Action:**  
1. Free up space immediately:  
   ```bash
   df -h # Identify full partitions
   sudo rm -rf /path/to/large/files # Remove unnecessary files
   ```
2. Investigate root cause: Check cron jobs, backups, or misconfigured services consuming disk space.  
3. Implement quotas via `/etc/fstab` or cloud provider console (AWS EBS, Azure Disks).  

---

#### **Critical Alert 3: LOG1003 & LOG1001**  
**UUIDs:** `9d3b7e2f-a3f9-4c1e-8b6c-7e2fa3f98b6c` (LOG1003), `a3f9d2b7-4c1e-4f8a-9d3b-7e2f1a6c8b9d` (LOG1001)  
**Why Critical:** Failed log transmissions to syslog servers (`syslog01`) break audit trails. This hinders incident detection, forensics, and compliance (e.g., GDPR, HIPAA).  

**Action:**  
1. Verify syslog server health:  
   ```bash
   systemctl status rsyslog # Linux
   netstat -an | grep 514 # Check port 514 UDP/TCP
   ```
2. Increase buffer sizes in `/etc/rsyslog.conf`:  
   ```conf
   $ActionFileEnableExec on
   $ActionFileDefaultTemplate RSYSLOG_TraditionalFileFormat
   ```
3. Use cloud-native solutions like AWS CloudWatch Logs or Azure Monitor Diagnostics.  

---

#### **Critical Alert 4: SYS2010**  
**UUID:** `4c1e9d3b-7e2f-4f8a-9d3b-7e2fa3f98b6c`  
**Why Critical:** Dependency cycles cause deadlocks, crashing critical services. This impacts application uptime and data processing.  

**Action:**  
1. Break the cycle using process managers (e.g., systemd):  
   ```ini
   [Service]
   KillMode=process
   Restart=on-failure
   ```
2. Review code for circular dependencies in deployment pipelines (e.g., Kubernetes Jobs).  
3. Enable deadlock detection in OS kernel parameters:  
   ```bash
   echo "kernel.yama.ptrace_scope=0" >> /etc/sysctl.conf
   sysctl -p
   ```

---

### Non-Critical Alerts (Require Investigation but Not Immediate Action)  
1. **SEC3003** (SSH DoS Attack):  
   - **Priority:** Medium. Requires blocking the attacker’s IP via firewall rules.  
   - **Action:** Update IPS/IDS signatures; analyze traffic patterns.  

2. **SEC3006** (Account Lockout):  
   - **Priority:** Low-Medium. Likely a brute-force attempt.  
   - **Action:** Review authentication logs; adjust lockout thresholds.  

3. **LOG1002** (Full Partition):  
   - **Priority:** Medium. Temporary workaround: Delete old logs via `journalctl --vacuum-size`.  

4. **REP5001** (Report Failure):  
   - **Priority:** Low. Retry manually; check scheduler permissions.  

5. **APP4005** (Segmentation Fault):  
   - **Priority:** Medium. Debug with GDB or Valgrind; update libraries.  

---

### Summary of Actions  
| Alert ID               | Severity | Immediate Steps                                                                 |
|------------------------|----------|----------------------------------------------------------------------------------|
| SEC3007                | Critical | Revoke SSH entry; audit configs                                                |
| SYS2002                | Critical | Free disk space; enable quotas                                                 |
| LOG1003 & LOG1001      | Critical | Fix syslog server connectivity; increase buffers                                |
| SYS2010                | Critical | Break dependency cycles; enable deadlock detection                               |

**Final Recommendation:** Prioritize resolving **SEC3007**, **SYS2002**, **LOG1003**, and **SYS2010** within 60 minutes to prevent cascading failures. Document all changes per ITIL standards.